<a href="https://colab.research.google.com/github/Sav200512/sp500-rfm-segmentation-cw/blob/main/Task_1_and_2_cw_(20232749_W2151870).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Task 1 Data Preparation and Feature Engineering**

---


**Task 1.1 Ticker selection and data acquisition**



In [92]:
import os, json, platform
import numpy as np
import pandas as pd
import yfinance as yf

In [93]:
SEED = 42
START, END = "2021-01-01", "2025-01-01"   # END is exclusive
BENCH = "^GSPC"

for folder in ["data/raw", "data/processed", "figures"]:
    os.makedirs(folder, exist_ok=True)

print("pandas", pd.__version__, "| numpy", np.__version__, "| yfinance", yf.__version__)

pandas 2.2.3 | numpy 2.4.6 | yfinance 0.2.66


In [94]:
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
table = pd.read_html(url, storage_options={"User-Agent": "Mozilla/5.0"})
sp500_df = table[0]

sp500_df["Yahoo_Symbol"] = sp500_df["Symbol"].str.replace(".", "-", regex=False)
sp500_df["Retrieved_On"] = pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d %H:%M UTC")

sp500_df[["Symbol", "Yahoo_Symbol", "Security", "GICS Sector", "Retrieved_On"]].to_csv(
    "data/raw/sp500_tickers.csv", index=False)

tickers = sp500_df["Yahoo_Symbol"].tolist()
print("Total tickers extracted:", len(tickers), "| retrieved:", sp500_df["Retrieved_On"].iloc[0])
sp500_df[["Symbol", "Security", "GICS Sector"]].head(10)

Total tickers extracted: 503 | retrieved: 2026-11-11 11:08 UTC


,Symbol,Security,GICS Sector
0,MMM,3M,Industrials
1,AOS,A. O. Smith,Industrials
2,ABT,Abbott Laboratories,Health Care
3,ABBV,AbbVie,Health Care
4,ACN,Accenture,Information Technology
5,ADBE,Adobe Inc.,Information Technology
6,AMD,Advanced Micro Devices,Information Technology
7,AES,AES Corporation,Utilities
8,AFL,Aflac,Financials
9,A,Agilent Technologies,Health Care


In [95]:
frames = []
for i in range(0, len(tickers), 50):
    batch = tickers[i:i + 50]
    d = yf.download(batch, start=START, end=END, auto_adjust=False,
                    progress=False, threads=True)
    frames.append(d)
    print(f"Batch {i // 50 + 1} done ({len(batch)} tickers)")

raw = pd.concat(frames, axis=1)
close, adj_close, volume = raw["Close"], raw["Adj Close"], raw["Volume"]

bench = yf.download(BENCH, start=START, end=END, auto_adjust=False, progress=False)
if isinstance(bench.columns, pd.MultiIndex):
    bench.columns = bench.columns.get_level_values(0)

Batch 1 done (50 tickers)
Batch 2 done (50 tickers)
Batch 3 done (50 tickers)


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['FDXF']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


Batch 4 done (50 tickers)


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['HONA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


Batch 5 done (50 tickers)
Batch 6 done (50 tickers)
Batch 7 done (50 tickers)


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['Q']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


Batch 8 done (50 tickers)


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['SNDK']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


Batch 9 done (50 tickers)


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['VYLR']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


Batch 10 done (50 tickers)
Batch 11 done (3 tickers)


In [96]:
close.to_csv("data/raw/close_daily.csv.gz")
adj_close.to_csv("data/raw/adj_close_daily.csv.gz")
volume.to_csv("data/raw/volume_daily.csv.gz")
bench.to_csv("data/raw/gspc_daily.csv")

run_log = {
    "download_time_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "start": START, "end_exclusive": END, "benchmark": BENCH,
    "yfinance": yf.__version__, "pandas": pd.__version__,
    "python": platform.python_version(),
    "tickers_requested": len(tickers),
}
json.dump(run_log, open("data/raw/run_log.json", "w"), indent=2)

no_data = [t for t in tickers if t not in adj_close.columns or adj_close[t].isna().all()]
print("Requested:", len(tickers))
print("Returned with some data:", len(tickers) - len(no_data))
print("No data:", no_data)
print("Shares: first date", adj_close.index.min().date(), "| last date", adj_close.index.max().date())
print("Benchmark: first date", bench.index.min().date(), "| last date", bench.index.max().date(), "| rows", len(bench))
print("Share rows:", len(adj_close), "| shape close/adj/vol:", close.shape, adj_close.shape, volume.shape)


Requested: 503
Returned with some data: 498
No data: ['FDXF', 'HONA', 'Q', 'SNDK', 'VYLR']
Shares: first date 2021-01-04 | last date 2024-12-31
Benchmark: first date 2021-01-04 | last date 2024-12-31 | rows 1005
Share rows: 1005 | shape close/adj/vol: (1005, 503) (1005, 503) (1005, 503)


**Task 1.2 Data validation and alignment**

---



In [97]:
# Index-level checks
print("Duplicate dates (shares):", adj_close.index.duplicated().sum())
print("Duplicate dates (benchmark):", bench.index.duplicated().sum())
print("Dates sorted:", adj_close.index.is_monotonic_increasing)
print("Share and benchmark dates identical:", adj_close.index.equals(bench.index))

# Per-share diagnostics
cols = adj_close.columns
diag = pd.DataFrame(index=cols)
diag["first_valid"]     = adj_close.apply(lambda s: s.first_valid_index())
diag["last_valid"]      = adj_close.apply(lambda s: s.last_valid_index())
diag["n_obs"]           = adj_close.notna().sum()
diag["miss_adj"]        = adj_close.isna().sum()
diag["miss_close"]      = close[cols].isna().sum()
diag["miss_vol"]        = volume[cols].isna().sum()
diag["nonpos_price"]    = ((adj_close <= 0) | (close[cols] <= 0)).sum()
diag["neg_vol"]         = (volume[cols] < 0).sum()
diag["zero_vol"]        = (volume[cols] == 0).sum()

print("Trading sessions:", len(adj_close))
print(diag.describe().T[["min", "max"]])
diag[diag["n_obs"] < len(adj_close)].sort_values("n_obs")

Duplicate dates (shares): 0
Duplicate dates (benchmark): 0
Dates sorted: True
Share and benchmark dates identical: True
Trading sessions: 1005
                              min                  max
first_valid   2021-01-04 00:00:00  2024-03-27 00:00:00
last_valid    2024-12-31 00:00:00  2024-12-31 00:00:00
n_obs                         0.0               1005.0
miss_adj                      0.0               1005.0
miss_close                    0.0               1005.0
miss_vol                      0.0               1005.0
nonpos_price                  0.0                  0.0
neg_vol                       0.0                  0.0
zero_vol                      0.0                424.0


,first_valid,last_valid,n_obs,miss_adj,miss_close,miss_vol,nonpos_price,neg_vol,zero_vol
Ticker,,,,,,,,,
HONA,NaT,NaT,0,1005,1005,1005,0,0,0
FDXF,NaT,NaT,0,1005,1005,1005,0,0,0
SNDK,NaT,NaT,0,1005,1005,1005,0,0,0
Q,NaT,NaT,0,1005,1005,1005,0,0,0
VYLR,NaT,NaT,0,1005,1005,1005,0,0,0
GEV,2024-03-27,2024-12-31,193,812,812,812,0,0,0
SOLV,2024-03-26,2024-12-31,194,811,811,811,0,0,0
RDDT,2024-03-21,2024-12-31,197,808,808,808,0,0,0
VLTO,2023-10-04,2024-12-31,313,692,692,692,0,0,0


In [98]:
n_sessions = len(adj_close)
reasons = {}

for t, r in diag.iterrows():
    why = []
    if r["n_obs"] == 0:
        why.append("no price data in study period")
    elif r["n_obs"] < n_sessions:
        why.append(f"incomplete history: first price {r['first_valid'].date()} ({int(r['n_obs'])}/{n_sessions} sessions)")
    elif r["miss_close"] > 0 or r["miss_vol"] > 0:
        why.append("missing Close or Volume")
    if r["nonpos_price"] > 0:
        why.append("nonpositive price")
    if r["neg_vol"] > 0:
        why.append("negative volume")
    if why:
        reasons[t] = "; ".join(why)

exclusion_log = pd.DataFrame({"Ticker": list(reasons), "Reason": list(reasons.values())})
included = [t for t in cols if t not in reasons]

exclusion_log.to_csv("data/processed/exclusion_log.csv", index=False)
print("\nExclusion Log")
display(exclusion_log.head(20))
pd.Series(included, name="Ticker").to_csv("data/processed/included_shares.csv", index=False)


Exclusion Log


,Ticker,Reason
0,APP,incomplete history: first price 2021-04-15 (93...
1,CEG,incomplete history: first price 2022-01-19 (74...
2,COIN,incomplete history: first price 2021-04-14 (93...
3,EXE,incomplete history: first price 2021-02-10 (97...
4,FDXF,no price data in study period
5,GEHC,incomplete history: first price 2022-12-15 (51...
6,GEV,incomplete history: first price 2024-03-27 (19...
7,HONA,no price data in study period
8,HOOD,incomplete history: first price 2021-07-29 (86...
9,Q,no price data in study period


In [99]:
validation_summary = pd.DataFrame({
    "Rule": [
        "No data available",
        "Incomplete history",
        "Non-positive prices",
        "Negative volume"
    ],
    "Count": [
        (diag["n_obs"] == 0).sum(),
        ((diag["n_obs"] > 0) &
         (diag["n_obs"] < n_sessions)).sum(),
        (diag["nonpos_price"] > 0).sum(),
        (diag["neg_vol"] > 0).sum()
    ]
})

validation_summary.to_csv(
    "data/processed/validation_summary.csv",
    index=False
)

print("\nValidation Summary")
display(validation_summary)

print("Included:", len(included), "| Excluded:", len(exclusion_log))
print(exclusion_log["Reason"].str.split(":").str[0].value_counts())
print("Shares with any zero-volume days (not excluded, check):", int((diag.loc[included, "zero_vol"] > 0).sum()))


Validation Summary


,Rule,Count
0,No data available,5
1,Incomplete history,10
2,Non-positive prices,0
3,Negative volume,0


Included: 488 | Excluded: 15
Reason
incomplete history               10
no price data in study period     5
Name: count, dtype: int64
Shares with any zero-volume days (not excluded, check): 2


Exclusion Rationale

Shares with no available prices during the study period were excluded. Shares with incomplete histories were also excluded because they lack observations for part of the 2021–2024 period. Retaining such shares would produce unequal observation windows and inconsistent weekly return counts, reducing comparability across securities.

In [100]:
# Large one-day moves in Close can signal an unadjusted split or a data error.
close_ret = close.pct_change(fill_method=None)
big = (close_ret.abs() > 0.40).sum()
big = big[big > 0].sort_values(ascending=False)
print("Shares with a one-day Close move > 40%:", len(big))
print(big)

# Compare Adj Close and Close.
# Differences arise from corporate-action adjustments
# such as dividends, stock splits and spin-offs.
ratio = adj_close / close[cols]
print("Shares where Adj Close differs from Close on at least one date:", (ratio.sub(1).abs().max() > 1e-9).sum())

# Spot-check a known split (example: NVDA 10-for-1, June 2024)
print(close["NVDA"].loc["2024-06-05":"2024-06-14"])
print(adj_close["NVDA"].loc["2024-06-05":"2024-06-14"])

print("\nNVDA corporate actions:")
print(yf.Ticker("NVDA").actions.loc["2024"])

Shares with a one-day Close move > 40%: 7
Ticker
CVNA    5
APP     1
BE      1
DXCM    1
GL      1
HOOD    1
RDDT    1
dtype: int64
Shares where Adj Close differs from Close on at least one date: 405
Date
2024-06-05    122.440002
2024-06-06    120.998001
2024-06-07    120.888000
2024-06-10    121.790001
2024-06-11    120.910004
2024-06-12    125.199997
2024-06-13    129.610001
2024-06-14    131.880005
Name: NVDA, dtype: float64
Date
2024-06-05    122.091866
2024-06-06    120.653961
2024-06-07    120.544289
2024-06-10    121.443718
2024-06-11    120.576118
2024-06-12    124.854271
2024-06-13    129.252106
2024-06-14    131.515839
Name: NVDA, dtype: float64

NVDA corporate actions:
                           Dividends  Stock Splits
Date                                              
2024-03-05 00:00:00-05:00      0.004           0.0
2024-06-10 00:00:00-04:00      0.000          10.0
2024-06-11 00:00:00-04:00      0.010           0.0
2024-09-12 00:00:00-04:00      0.010           0.0
2024-

In [101]:
adj_inc = adj_close[included]
bench_adj = bench["Adj Close"]

w_shares = adj_inc.resample("W-FRI").last()
w_bench  = bench_adj.resample("W-FRI").last()

# Drop incomplete first/last calendar weeks (Mon-Fri window must sit inside the study period)
last_date = adj_close.index.max()
week_start = w_shares.index - pd.Timedelta(days=4)
keep = (week_start >= pd.Timestamp(START)) & (w_shares.index <= last_date)
w_shares, w_bench = w_shares[keep], w_bench[keep]

OBSERVATION_DATE = last_date

print(
    "Final common trading date:",
    OBSERVATION_DATE.date()
)

assert w_shares.index.equals(w_bench.index), "Week labels differ"

ret_shares = w_shares.pct_change(fill_method=None).iloc[1:]
ret_bench  = w_bench.pct_change(fill_method=None).iloc[1:]

ret_shares.to_csv(
    "data/processed/weekly_share_returns.csv.gz"
)

ret_bench.to_csv(
    "data/processed/weekly_benchmark_returns.csv"
)

print("Weekly price labels:", w_shares.index.min().date(), "to", w_shares.index.max().date(), f"({len(w_shares)})")
print("Weekly return observations:", len(ret_shares))
print("Any NaN in share returns:", ret_shares.isna().any().any())

Final common trading date: 2024-12-31
Weekly price labels: 2021-01-08 to 2024-12-27 (208)
Weekly return observations: 207
Any NaN in share returns: False


In [102]:
# 1. Zero-volume shares among the included set
zv = diag.loc[included, "zero_vol"]
print("Included shares with zero-volume days:")
for t in zv[zv > 0].index:
    v = volume[t]
    z = v[v == 0]

    print(
        f"  {t} | zero days: {len(z)} "
        f"| first: {z.index.min().date()} "
        f"| last: {z.index.max().date()} "
        f"| median non-zero volume: {v[v > 0].median():,.0f}"
    )

print("\nPotential volume anomalies:")
print(
    diag.loc[
        zv[zv > 20].index,
        ["zero_vol", "n_obs"]
    ]
)

# 2. The one-day moves over 40%
print("\nMoves over 40%:")
for t in big.index:
    moves = close_ret[t][close_ret[t].abs() > 0.40]
    print(f"\n{t} | included: {t in included}")
    print(moves.round(3).to_string())
    try:
        sp = yf.Ticker(t).splits
        print("  Yahoo splits 2021-2024:", sp.loc["2021":"2024"].to_dict() if len(sp) else "none")
    except Exception as e:
        print("  splits lookup failed:", e)

# 3. Late listings: any gaps after the first price?
late = diag[(diag["n_obs"] > 0) & (diag["n_obs"] < n_sessions)].copy()
late["gaps_after_first_price"] = [int(adj_close.loc[late.loc[t, "first_valid"]:, t].isna().sum()) for t in late.index]
print("\nLate listings:")
print(late[["first_valid", "n_obs", "gaps_after_first_price"]])

# Save diagnostics
diag.to_csv(
    "data/processed/share_diagnostics.csv"
)

Included shares with zero-volume days:
  FERG | zero days: 27 | first: 2021-01-07 | last: 2021-03-05 | median non-zero volume: 803,550
  SW | zero days: 424 | first: 2021-01-05 | last: 2024-07-05 | median non-zero volume: 600

Potential volume anomalies:
        zero_vol  n_obs
Ticker                 
FERG          27   1005
SW           424   1005

Moves over 40%:

CVNA | included: True
Date
2022-08-05    0.401
2022-12-07   -0.429
2023-01-12    0.460
2023-06-08    0.560
2023-07-19    0.402
  Yahoo splits 2021-2024: {}

APP | included: False
Date
2024-11-07    0.463
  Yahoo splits 2021-2024: none

BE | included: True
Date
2024-11-15    0.592
  Yahoo splits 2021-2024: none

DXCM | included: True
Date
2024-07-26   -0.407
  Yahoo splits 2021-2024: {Timestamp('2022-06-13 00:00:00-0400', tz='America/New_York'): 4.0}

GL | included: True
Date
2024-04-11   -0.531
  Yahoo splits 2021-2024: {}

HOOD | included: False
Date
2021-08-04    0.504
  Yahoo splits 2021-2024: none

RDDT | included: Fals

Survivorship Bias

The S&P 500 constituent list was obtained from the current Wikipedia S&P 500 page rather than a historical constituent database covering 2021–2024. As a result, survivorship bias may be present. Firms removed from the index during the study period are not represented, while companies added later are included. Therefore the sample reflects current constituents rather than the exact historical composition of the S&P 500 throughout the study period.

**Task 1.3**
---

**Task 1.3.1 Mean Weekly Return**

In [103]:
mean_weekly_return = ret_shares.mean()

print("Mean Weekly Return Summary")
print(mean_weekly_return.describe())

mean_weekly_return.to_csv(
    "data/processed/mean_weekly_return.csv",
    header=["Mean_Weekly_Return"]
)


Mean Weekly Return Summary
count    488.000000
mean       0.002857
std        0.002646
min       -0.004468
25%        0.001182
50%        0.002629
75%        0.004207
max        0.017772
dtype: float64


In [104]:
print(
    "Weekly observations used:",
    len(ret_shares)
)

print(
    "Any missing mean returns:",
    mean_weekly_return.isna().sum()
)

Weekly observations used: 207
Any missing mean returns: 0


**Task 1.3.2 weekly beta**

---

In [105]:
market_var = ret_bench.var()

weekly_beta = ret_shares.apply(
    lambda s: s.cov(ret_bench) / market_var
)

print("\nWeekly Beta Summary")
print(weekly_beta.describe())

weekly_beta.to_csv(
    "data/processed/weekly_beta.csv",
    header=["Weekly_Beta"]
)


Weekly Beta Summary
count    488.000000
mean       0.973868
std        0.406550
min        0.055452
25%        0.666634
50%        0.950610
75%        1.179289
max        3.778560
dtype: float64


In [106]:
risk_return_features = pd.DataFrame({
    "Ticker": mean_weekly_return.index,
    "Mean_Weekly_Return": mean_weekly_return.values,
    "Weekly_Beta": weekly_beta.values
})

print(
    risk_return_features.describe()
)

risk_return_features.to_csv(
    "data/processed/risk_return_features.csv",
    index=False
)

       Mean_Weekly_Return  Weekly_Beta
count          488.000000   488.000000
mean             0.002857     0.973868
std              0.002646     0.406550
min             -0.004468     0.055452
25%              0.001182     0.666634
50%              0.002629     0.950610
75%              0.004207     1.179289
max              0.017772     3.778560


Weekly Return and Weekly Beta

Weekly returns were calculated using Friday-ending adjusted closing prices and the percentage change from the previous completed week. The arithmetic mean weekly return over the full study period was used as the return feature. Weekly beta was calculated as the covariance between individual share returns and benchmark returns divided by the variance of benchmark returns. All calculations used 207 aligned weekly return observations and no missing weekly returns were imputed.

In [107]:
print(
    risk_return_features
    .sort_values(
        "Weekly_Beta",
        ascending=False
    )
    .head(10)
)


    Ticker  Mean_Weekly_Return  Weekly_Beta
94    CVNA            0.010964     3.778560
98     XYZ           -0.001554     2.504520
114   COHR            0.003887     2.139305
331   NVDA            0.013624     2.125953
440   TWLO           -0.002161     2.073379
432   TSLA            0.005275     2.045761
61      BE            0.003802     2.017079
361   PLTR            0.009613     2.003194
460    VRT            0.012583     2.001480
82     CCL            0.004095     1.958835


In [108]:
print("\nAligned weekly observations used:", len(ret_shares))
print("Missing returns:", risk_return_features.isna().sum().sum())


Aligned weekly observations used: 207
Missing returns: 0


**Task 1.4 Adapted RFM Calculations**

---



In [115]:
OBSERVATION_DATE = last_date
print(last_date)

2024-12-31 00:00:00


In [120]:
#RFM
#Volume > 2 * mean(previous 20 trading sessions)

Vol_inc = volume[included]
close_inc = close[included]

baseline20 = vol_inc.shift(1).rolling(
    20,
    min_periods=20
).mean()



In [123]:
event_flag = vol_inc > (2 * baseline20)
True
#hight-volume event

print(
    "Total events:",
    event_flag.sum().sum()
)

Total events: 18496


**1.4.1 Recency**

---



In [124]:
#find most recent event
#observation date - event date

recency = {}

for ticker in included:

    events = event_flag.index[event_flag[ticker]]

    if len(events) == 0:
        recency[ticker] = np.nan
    else:
        recency[ticker] = (
            OBSERVATION_DATE -
            events.max()
        ).days

recency = pd.Series(
    recency,
    name="R"
)


In [125]:
print(recency.describe())
print(
    "No event stocks:",
    recency.isna().sum()
)

count    488.000000
mean      20.389344
std       21.302843
min        4.000000
25%       11.000000
50%       11.000000
75%       15.000000
max      222.000000
Name: R, dtype: float64
No event stocks: 0


**Task 1.4.2 Frequency**

---



In [126]:
#Use the latest 60 trading sessions ending on observation date

last60_events = event_flag.tail(60)
frequency = last60_events.sum()

frequency.name = "F"

print(frequency.describe())


count    488.000000
mean       2.987705
std        1.328820
min        0.000000
25%        2.000000
50%        3.000000
75%        4.000000
max        8.000000
Name: F, dtype: float64


**Task 1.4.3 Monetary**

---

In [127]:
# close*Volume
#sum of last 60 sessions

last60_close = close_inc.tail(60)
last60_vol = vol_inc.tail(60)

daily_traded_value = (
    last60_close *
    last60_vol
)

monetary = daily_traded_value.sum()

monetary.name = "M"

print(monetary.describe())

count    4.880000e+02
mean     3.930290e+10
std      1.288833e+11
min      1.297483e+09
25%      1.023194e+10
50%      1.668382e+10
75%      3.210048e+10
max      1.847697e+12
Name: M, dtype: float64


In [131]:
#RFM TABLE
rfm_features = pd.concat(
    [
        recency,
        frequency,
        monetary
    ],
    axis=1
)

In [132]:
print(
    "Rows:",
    len(rfm_features)
)

print(
    "Missing values:"
)
print(
    rfm_features.isna().sum()
)

Rows: 488
Missing values:
R    0
F    0
M    0
dtype: int64


In [133]:
rfm_features.to_csv(
    "data/processed/rfm_features.csv"
)

In [134]:
print(
    rfm_features.loc["AAPL"]
)

R    1.100000e+01
F    2.000000e+00
M    6.233034e+11
Name: AAPL, dtype: float64
